# Step 4e — Purchase-sequence affinity (feature construction only)

**Question:** Does this candidate belong to a level-2 category often purchased after a category this user recently purchased, beyond destination-category popularity?

Following Step 4b, this notebook contains the entire workflow: setup, feature derivation, implementation, correctness checks, candidate-aligned construction, diagnostics and safe saving. **No custom Python modules or external test files are needed.** Run the cells from top to bottom.

**Inputs:** `user_behavior_data.txt` and `product_meta_data.txt`. No pre-existing Parquet inputs are required. An existing `processed/features.parquet`, if present, supplies authoritative splits and must match candidate occurrence order.

**Outputs:** separate `processed/purchase_sequence_affinity_sample/` artifacts. The main feature table is unchanged. Model training and ranking evaluation are deferred. Anonymized category associations are observational, not verified complements or causal effects.


In [1]:
import os, csv, hashlib, json, math, platform, resource, tempfile, time, unittest
from collections import Counter
from dataclasses import asdict, dataclass, replace
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', Path.cwd() / 'Raw Data')).expanduser().resolve()

OUT_DIR = DATA_DIR / 'processed' / 'purchase_sequence_affinity_sample'
NOTEBOOK_PATH = Path.cwd() / 'step4e_purchase_sequence_affinity.ipynb'
SAMPLE_SIZE, SEED = 20_000, 42
REBUILD = True  # False verifies and inspects an existing completed run.
FEATURE_COLUMNS = ['sequence_score', 'sequence_support_log', 'sequence_source_age_days', 'sequence_available']
ASSOCIATION_COLUMNS = ['source_category', 'destination_category', 'source_episodes',
    'pair_episodes', 'pair_records', 'probability', 'background_probability', 'lift',
    'support_threshold_met', 'positive_lift', 'supported']
print('Input directory:', DATA_DIR)
for name in ['user_behavior_data.txt', 'product_meta_data.txt']:
    assert (DATA_DIR / name).is_file(), name


Input directory: /Users/chounrayhan/Desktop/Repos/BT4222/BT4222-Group7/Raw Data


## 1. Configuration and safe output helpers

The defaults below follow the handoff exactly. Keep them fixed for the first experiment. Files are written to temporary paths and verified before replacement. The manifest is written last as the completion marker. Provenance hashes notebook **code cells only**, so saving execution outputs does not invalidate the implementation hash.


In [2]:
@dataclass(frozen=True)
class Config:
    episode_minutes: float = 30
    min_delay_hours: float = 1
    max_delay_days: float = 7
    buffer_days: float = 1
    alpha: float = 20
    min_source_episodes: int = 50
    min_pair_records: int = 10
    recent_days: float = 30
    decay_days: float = 7

    def __post_init__(self):
        if not all(math.isfinite(v) for v in asdict(self).values()):
            raise ValueError("Configuration must be finite")
        if (self.episode_minutes < 0 or self.min_delay_hours < 0
                or self.max_delay_days < self.min_delay_hours / 24
                or self.buffer_days < 0 or self.alpha <= 0
                or self.min_source_episodes < 1 or self.min_pair_records < 1
                or self.recent_days < 0 or self.decay_days <= 0):
            raise ValueError("Invalid feature configuration")


In [3]:
def fingerprint(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return dict(path=str(path.resolve()), size_bytes=path.stat().st_size, sha256=h.hexdigest())

def save_json(value, path):
    tmp = path.with_suffix('.tmp.json')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False) + '\n')
    json.loads(tmp.read_text())
    os.replace(tmp, path)

def save_parquet(frame, path):
    tmp = path.with_suffix('.tmp.parquet')
    frame.to_parquet(tmp, index=False)
    assert pq.ParquetFile(tmp).metadata.num_rows == len(frame)
    os.replace(tmp, path)

def notebook_fingerprint(path=NOTEBOOK_PATH):
    notebook = json.loads(path.read_text())
    code = '\0'.join(''.join(c['source']) for c in notebook['cells'] if c['cell_type'] == 'code')
    return dict(path=str(path.resolve()), kind='notebook_code', sha256=hashlib.sha256(code.encode()).hexdigest())


## 1. Data preparation and alignment

Uniformly select 20,000 original record indices without replacement, seed 42, across the entire behavior file. Keep complete histories/slates and use the original zero-based record index as `user`, matching Step 1. It is a record/session key, not a verified persistent customer ID.

When no feature table exists, reproduce Step 2's seed-42 70/15/15 draw for **all** original records before selecting sampled records. Never redraw splits on the sample. If the main feature table exists, reuse its assignments and reject incompatible candidate occurrence alignment.

Read the TXT files directly. Stream only `wid` and `cate_id_2` metadata and retain needed products. Preserve integer identifiers, validate history alignment, reject conflicting metadata for needed IDs, and exclude null/negative/missing categories. `candidate_pos` is retained solely as an occurrence key; it is excluded from predictors and scoring tie-breaking.


In [4]:
def reconstruct_ages(gaps, n_events):
    """Step 1 semantics, plus validation; returned order is earliest to latest."""
    import numpy as np
    gaps = np.asarray(gaps, dtype=np.float64)
    if (len(gaps) != n_events + 1 or gaps[0] != 0
            or not np.isfinite(gaps).all() or (gaps < 0).any()):
        raise ValueError("History gaps must be finite, nonnegative, aligned, and start at zero")
    return np.cumsum(gaps[1:][::-1])[::-1] / 86400


In [5]:
def load_records(path, sample_size, seed):
    with path.open() as f:
        header = next(f).rstrip('\n').split('\t')
        total = sum(1 for _ in f)
    expected = ['query', 'candidate_wid_list', 'candidate_label_list', 'history_qry_list',
                'history_wid_list', 'history_type_list', 'history_time_list']
    if header != expected:
        raise ValueError(f'Unexpected behavior header: {header}')
    if total == 0:
        raise ValueError('Empty behavior dataset')
    selected = set(np.random.default_rng(seed).choice(total, min(sample_size, total), replace=False).tolist())
    # Match Step 2 exactly: draw for ALL original records before subsetting.
    splits = np.random.default_rng(42).choice(['train', 'val', 'test'], size=total, p=[.70, .15, .15])
    records, needed = [], set()
    with path.open(newline='') as f:
        reader = csv.DictReader(f, delimiter='\t', quoting=csv.QUOTE_NONE)
        for user, row in enumerate(reader):
            if user not in selected:
                continue
            wids = [int(x) for x in row['history_wid_list'].split('_')] if row['history_wid_list'] else []
            types = row['history_type_list'].split('_') if wids else []
            queries = row['history_qry_list'].split('_') if wids else []
            if len(wids) != len(types) or len(wids) != len(queries):
                raise ValueError(f'Misaligned history in original record {user}')
            if set(types) - {'CLICK', 'CART', 'ORD', 'FLW'}:
                raise ValueError(f'Unknown history action in record {user}')
            ages = reconstruct_ages([float(x) for x in row['history_time_list'].split('_')], len(wids))
            orders = [(wid, float(age)) for wid, typ, age in zip(wids, types, ages) if typ == 'ORD']
            candidates = [int(x) for x in row['candidate_wid_list'].split('_')]
            # Labels are retained for alignment only, never passed to fit/transform.
            labels = [float(x) for x in row['candidate_label_list'].split('_')]
            if len(candidates) != len(labels) or any(x not in (0, 1, 2, 3) for x in labels):
                raise ValueError(f'Invalid candidate labels/alignment in record {user}')
            records.append(dict(user=user, split=str(splits[user]), raw_orders=orders,
                                candidates=candidates, labels=[int(x) for x in labels]))
            needed.update(candidates)
            needed.update(wid for wid, _ in orders)
    return records, np.array(sorted(needed), dtype=np.int64), total


In [6]:
def load_metadata(path, needed):
    """Stream only ID/category columns, retaining IDs needed by the sample."""
    reader = pacsv.open_csv(path,
        read_options=pacsv.ReadOptions(block_size=8 * 1024 * 1024),
        parse_options=pacsv.ParseOptions(delimiter='\t', quote_char=False),
        convert_options=pacsv.ConvertOptions(include_columns=['wid', 'cate_id_2'],
            column_types={'wid': pa.int64(), 'cate_id_2': pa.int64()}))
    metadata, seen, duplicate_count = {}, set(), 0
    for batch in reader:
        if batch['wid'].null_count:
            raise ValueError('Null metadata product ID')
        wid = batch['wid'].to_numpy()
        index = np.searchsorted(needed, wid)
        keep = (index < len(needed)) & (needed[np.minimum(index, len(needed) - 1)] == wid)
        small = pa.Table.from_batches([batch]).filter(pa.array(keep))
        for product, category in zip(small['wid'].to_pylist(), small['cate_id_2'].to_pylist()):
            category = None if category is None or category < 0 else category
            if product in seen:
                duplicate_count += 1
                if metadata.get(product) != category:
                    raise ValueError(f'Conflicting category assignments for product {product}')
            seen.add(product)
            metadata[product] = category
    return metadata, duplicate_count

def use_existing_splits(records, feature_path):
    """Honor an existing feature table, checking occurrence order and labels."""
    if not feature_path.exists():
        return 'Step 2 seed=42, 70/15/15 draw across all original record IDs'
    import pyarrow.dataset as ds
    table = ds.dataset(feature_path).to_table(columns=['user', 'wid', 'label', 'split'],
        filter=ds.field('user').isin([r['user'] for r in records])).to_pandas()
    by_user = {int(u): g for u, g in table.groupby('user', sort=False)}
    for record in records:
        g = by_user.get(record['user'])
        if (g is None or g['wid'].tolist() != record['candidates']
                or g['label'].tolist() != record['labels'] or g['split'].nunique() != 1):
            raise ValueError('Existing features have incompatible candidate occurrence alignment/splits')
        split = str(g['split'].iloc[0])
        if split not in ('train', 'val', 'test'):
            raise ValueError(f'Unknown split {split}')
        record['split'] = split
    return f'Existing feature table: {feature_path}'


In [7]:
# Hand-check Step 1's age semantics, including the final query gap.
print('Ages in seconds:', reconstruct_ages([0, 12, 4, 5], 3) * 86400)
assert list(reconstruct_ages([0, 12, 4, 5], 3) * 86400) == [21, 9, 5]


Ages in seconds: [21.  9.  5.]


## 2. Purchase episodes and association fitting

Use only `ORD` history actions (Step 1's numeric code is 2; candidate purchase labels are 3). Each category gets fixed 30-minute episodes anchored at its first purchase; represent an episode by its last purchase. The window does not roll forward. Sort episodes across categories, preserving stored order at equal times.

For each eligible source episode A, count distinct destination categories B occurring 1 hour–7 days later, inclusive, excluding A itself. Require source age ≥ 8 days: complete 7-day follow-up plus a 1-day pre-query buffer. Include empty source windows in denominators. Count distinct supporting original records separately from supporting episodes.

With `alpha = 20`, estimate `p_B = n_B / N`, `p_B_given_A = (n_AB + alpha*p_B)/(n_A + alpha)`, and `lift = p_B_given_A/p_B`. Only score pairs with ≥50 source episodes, ≥10 supporting training records, and lift >1.

Fit three association tables on two-thirds of training records to create out-of-fold training features (balanced random folds, seed 43). Fit a fourth table on all training records for validation/test. Held-out histories can describe the current session, but cannot update these tables. Candidate labels never enter fitting or transformation.


In [8]:
def episodes(orders, config=Config()):
    """orders: chronological (category, age_days); missing categories excluded.

    A fixed per-category anchor starts each episode; its representative is the
    last included event. Stable event index resolves equal representative times.
    """
    active, result = {}, []
    previous = math.inf
    for index, (category, age) in enumerate(orders):
        if not math.isfinite(age) or age < 0 or age > previous:
            raise ValueError("Orders must have finite non-increasing ages")
        previous = age
        if category is None:
            continue
        if category in active:
            anchor, last_age, last_index = active[category]
            if anchor - age <= config.episode_minutes / 1440 + 1e-12:
                active[category] = (anchor, age, index)
                continue
            result.append((category, last_age, last_index))
        active[category] = (age, age, index)
    result.extend((cat, age, index) for cat, (_, age, index) in active.items())
    result.sort(key=lambda x: (-x[1], x[2]))
    return [(cat, age) for cat, age, _ in result]


In [9]:
def count_associations(histories, config=Config()):
    """Count source windows, including empty windows, in distinct records.

    histories is an iterable of chronological order lists, one per original
    record. No labels, query text, or candidate products are accepted here.
    """
    sources, pairs, destinations, records = Counter(), Counter(), Counter(), Counter()
    for orders in histories:
        ep = episodes(orders, config)
        record_pairs = set()
        for i, (source, age) in enumerate(ep):
            if age + 1e-12 < config.max_delay_days + config.buffer_days:
                continue
            sources[source] += 1
            seen = set()
            for j in range(i + 1, len(ep)):
                dest, dest_age = ep[j]
                delay = age - dest_age
                if delay > config.max_delay_days + 1e-12:
                    break
                if delay + 1e-12 >= config.min_delay_hours / 24 and dest != source:
                    seen.add(dest)
            for dest in seen:
                pairs[source, dest] += 1
                destinations[dest] += 1
                record_pairs.add((source, dest))
        records.update(record_pairs)
    return dict(sources=sources, pairs=pairs, destinations=destinations, records=records)


In [10]:
def association_rows(counts, config=Config()):
    """All observed pairs with explicit support and positive-lift flags."""
    total = sum(counts['sources'].values())
    for (source, dest), n_ab in sorted(counts['pairs'].items()):
        n_a, u_ab = counts['sources'][source], counts['records'][source, dest]
        p_b = counts['destinations'][dest] / total if total else 0
        if p_b <= 0:
            continue
        probability = (n_ab + config.alpha * p_b) / (n_a + config.alpha)
        lift = probability / p_b
        support = n_a >= config.min_source_episodes and u_ab >= config.min_pair_records
        yield dict(source_category=source, destination_category=dest,
                   source_episodes=n_a, pair_episodes=n_ab, pair_records=u_ab,
                   probability=probability, background_probability=p_b, lift=lift,
                   support_threshold_met=support, positive_lift=lift > 1,
                   supported=support and lift > 1)

def scoring_lookup(rows):
    return {(r['source_category'], r['destination_category']):
            (r['lift'], r['pair_records']) for r in rows if r['supported']}

def fit_table(histories, out_dir, name, config):
    counts = count_associations(histories, config)
    rows = list(association_rows(counts, config))
    frame = pd.DataFrame(rows, columns=ASSOCIATION_COLUMNS)
    save_parquet(frame, out_dir / f'associations_{name}.parquet')
    # Source denominators include categories that never produce a pair.
    sources = pd.DataFrame(sorted(counts['sources'].items()), columns=['source_category', 'source_episodes'])
    save_parquet(sources, out_dir / f'source_windows_{name}.parquet')
    diag = dict(eligible_windows=sum(counts['sources'].values()), observed_pairs=len(rows),
                supported_positive_pairs=sum(r['supported'] for r in rows))
    print(f'{name}: {diag}', flush=True)
    return scoring_lookup(rows), diag


In [11]:
# Fixed anchor: the purchase at minute 40 starts a new episode.
example_orders = [(1, 10), (1, 10 - 20/1440), (1, 10 - 40/1440)]
print('Fixed-anchor episodes:', episodes(example_orders))
assert len(episodes(example_orders)) == 2
print('Configuration:', Config())


Fixed-anchor episodes: [(1, 9.98611111111111), (1, 9.972222222222221)]
Configuration: Config(episode_minutes=30, min_delay_hours=1, max_delay_days=7, buffer_days=1, alpha=20, min_source_episodes=50, min_pair_records=10, recent_days=30, decay_days=7)


## 3. Candidate scoring

For each candidate destination category, consider the user's most recent **actual order** in each source category within 30 days (inclusive), not episode representatives. Score each supported relationship with `exp(-source_age_days/7) * log(lift)` and choose the maximum. Exact float64 score ties select the smallest integer source category, independent of candidate order.

| Column | Meaning | No supported relationship |
|---|---|---|
| `sequence_score` | Maximum decayed positive log-lift | 0 |
| `sequence_support_log` | log1p(distinct training records supporting the selected relationship) | 0 |
| `sequence_source_age_days` | Most recent purchase age for the same selected source | missing |
| `sequence_available` | Whether a relationship was selected | 0 |

`sequence_candidate_category_available` is a separate diagnostic. Unknown candidate categories do not match each other. Products in the same destination category receive identical feature values within a session. Results are cached per destination category during scoring.


In [12]:
def transform(orders, destination_categories, lookup, config=Config()):
    """One tuple per candidate occurrence; repeated categories reuse scores.

    Equal scores select the smallest integer source category. Missing category
    is None, never a shared artificial category. Availability distinguishes a
    missing age from a meaningful age of zero.
    """
    recent = {}
    for cat, age in orders:
        if cat is not None and 0 <= age <= config.recent_days:
            recent[cat] = min(recent.get(cat, math.inf), age)
    cache = {}
    for dest in destination_categories:
        if dest not in cache:
            selected = (0.0, 0.0, math.nan, 0)
            if dest is not None:
                for source in sorted(recent):
                    if source == dest or (source, dest) not in lookup:
                        continue
                    lift, support = lookup[source, dest]
                    if lift <= 1:
                        continue
                    age = recent[source]
                    score = math.exp(-age / config.decay_days) * math.log(lift)
                    if score > selected[0]:
                        selected = (score, math.log1p(support), age, 1)
            cache[dest] = selected
        yield cache[dest]


In [13]:
# Independent hand calculation: lift 4, one-day-old purchase, 10 supporting records.
example = list(transform([(1, 1)], [2], {(1, 2): (4, 10)}))[0]
display(pd.DataFrame([example], columns=FEATURE_COLUMNS))
assert math.isclose(example[0], math.exp(-1/7) * math.log(4))
assert math.isclose(example[1], math.log1p(10))


,sequence_score,sequence_support_log,sequence_source_age_days,sequence_available
0,1.201748,2.397895,1,1


## 4. Candidate-aligned construction and diagnostics

This cell follows Step 4b's `NEW` dictionary convention, with one list per new feature, attached to the candidate rows without reordering. The full experiment is a notebook-local function so the same steps can also be checked on tiny synthetic TXT inputs below.

It samples original records, reuses or reproduces the established splits, fits out-of-fold training tables and the all-training table, scores candidates, checks alignment/value semantics, and saves the separate artifacts. `candidate_pos` and association fold IDs remain alignment/diagnostic fields and are excluded from predictors. Every build is fresh; no intermediate cache is silently reused.


In [14]:
def diagnostics(frame, records):
    result = {}
    for split in ['train', 'val', 'test']:
        f = frame[frame['split'] == split]
        rr = [r for r in records if r['split'] == split]
        order_total = sum(len(r['orders']) for r in rr)
        order_known = sum(cat is not None for r in rr for cat, _ in r['orders'])
        per_session = f.groupby('user')
        result[split] = dict(
            sessions=len(rr), candidate_rows=len(f), order_events=order_total,
            order_category_coverage=order_known / order_total if order_total else None,
            candidate_category_coverage=float(f['sequence_candidate_category_available'].mean()) if len(f) else None,
            candidate_availability=float(f['sequence_available'].mean()) if len(f) else None,
            session_availability=float(per_session['sequence_available'].max().mean()) if len(f) else None,
            session_score_variation=float((per_session['sequence_score'].nunique() > 1).mean()) if len(f) else None,
            feature_summary={c: {k: (None if pd.isna(v) else float(v))
                for k, v in f[c].describe().items()} for c in FEATURE_COLUMNS},
            missing_values={c: int(f[c].isna().sum()) for c in FEATURE_COLUMNS})
    return result


In [15]:
def build_experiment(data_dir, out_dir=None, sample_size=20_000, seed=42):
    started = time.perf_counter()
    data_dir = Path(data_dir).expanduser().resolve()
    out_dir = Path(out_dir) if out_dir else data_dir / 'processed' / 'purchase_sequence_affinity_sample'
    if sample_size < 1:
        raise ValueError('Sample size must be positive')
    config = Config()
    behavior = data_dir / 'user_behavior_data.txt'
    product = data_dir / 'product_meta_data.txt'
    # Every run rebuilds artifacts; fingerprints are provenance, not a stale cache key.
    print('Fingerprinting raw inputs...', flush=True)
    inputs = {p.name: fingerprint(p) for p in [behavior, product]}
    records, needed, total = load_records(behavior, sample_size, seed)
    existing = data_dir / 'processed' / 'features.parquet'
    split_policy = use_existing_splits(records, existing)
    if existing.exists():
        inputs['existing_features'] = fingerprint(existing)
    print(f'Selected {len(records):,}/{total:,} records; joining {len(needed):,} product IDs', flush=True)
    metadata, duplicates = load_metadata(product, needed)
    for r in records:
        r['orders'] = [(metadata.get(w), age) for w, age in r.pop('raw_orders')]
    train = [r for r in records if r['split'] == 'train']
    if len(train) < 3:
        raise ValueError('At least three training records required for cross-fitting')
    # Assign folds once on stable, ascending training record IDs.
    permutation = np.random.default_rng(seed + 1).permutation(len(train))
    fold_ids = np.empty(len(train), dtype=np.int8)
    fold_ids[permutation] = np.arange(len(train)) % 3
    for r, fold in zip(train, fold_ids):
        r['fold'] = int(fold)
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out_dir / 'manifest.json'
    # Remove completion marker before rebuilding so partial runs are detectable.
    manifest_path.unlink(missing_ok=True)
    tables, fit_diagnostics = {}, {}
    for fold in range(3):
        tables[fold], fit_diagnostics[f'fold_{fold}'] = fit_table(
            (r['orders'] for r in train if r['fold'] != fold), out_dir, f'fold_{fold}', config)
    tables['train'], fit_diagnostics['all_train'] = fit_table(
        (r['orders'] for r in train), out_dir, 'all_train', config)
    columns = ['user', 'candidate_pos', 'wid', 'label', 'split', 'association_fold',
               'sequence_candidate_category_available'] + FEATURE_COLUMNS
    output = []
    # Step 4b convention: one candidate-aligned array/list per new feature.
    NEW = {name: [] for name in FEATURE_COLUMNS}
    for r in records:
        cats = [metadata.get(w) for w in r['candidates']]
        table_key = r['fold'] if r['split'] == 'train' else 'train'
        for pos, (wid, label, cat, values) in enumerate(zip(r['candidates'], r['labels'], cats,
                transform(r['orders'], cats, tables[table_key], config))):
            output.append((r['user'], pos, wid, label, r['split'], r.get('fold', -1), int(cat is not None)))
            for name, value in zip(FEATURE_COLUMNS, values):
                NEW[name].append(value)
    frame = pd.DataFrame(output, columns=columns[:-len(FEATURE_COLUMNS)])
    for name, values in NEW.items():
        frame[name] = values
    del output, NEW
    for c in ['user', 'candidate_pos']:
        frame[c] = frame[c].astype('int32')
    for c in ['label', 'association_fold', 'sequence_available', 'sequence_candidate_category_available']:
        frame[c] = frame[c].astype('int8')
    for c in FEATURE_COLUMNS[:3]:
        frame[c] = frame[c].astype('float32')
    frame['wid'] = frame['wid'].astype('int64')
    assert not frame.duplicated(['user', 'candidate_pos']).any()
    assert len(frame) == sum(len(r['candidates']) for r in records)
    assert np.isfinite(frame['sequence_score']).all() and (frame['sequence_score'] >= 0).all()
    unavailable = frame['sequence_available'] == 0
    assert frame.loc[unavailable, 'sequence_source_age_days'].isna().all()
    assert (frame.loc[unavailable, ['sequence_score', 'sequence_support_log']] == 0).all().all()
    assert frame.loc[~unavailable, 'sequence_source_age_days'].between(0, config.recent_days).all()
    save_parquet(frame, out_dir / 'candidate_features.parquet')
    membership = pd.DataFrame([dict(user=r['user'], split=r['split'], association_fold=r.get('fold', -1)) for r in records])
    save_parquet(membership, out_dir / 'sample_membership.parquet')
    report = dict(input_records=total, selected_records=len(records),
        selected_order_events=sum(len(r['orders']) for r in records),
        unique_known_categories=len({c for c in metadata.values() if c is not None}),
        needed_product_ids=len(needed), matching_metadata_ids=len(metadata),
        duplicate_needed_metadata_rows=duplicates, fits=fit_diagnostics,
        by_split=diagnostics(frame, records), elapsed_seconds=time.perf_counter() - started,
        process_peak_rss_bytes=int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss *
                                   (1 if platform.system() == 'Darwin' else 1024)))
    save_json(report, out_dir / 'diagnostics.json')
    artifacts = {p.name: fingerprint(p) for p in sorted(out_dir.glob('*.parquet'))}
    artifacts['diagnostics.json'] = fingerprint(out_dir / 'diagnostics.json')
    manifest = dict(implementation_version=2, config=asdict(config), sample_size=sample_size,
        sample_seed=seed, split_policy=split_policy, crossfit_folds=3, fold_seed=seed + 1,
        tie_break='maximum float64 score; exact ties choose smallest integer source category',
        category='cate_id_2; negative/null/missing category excluded',
        keys=['user (original zero-based record index)', 'candidate_pos (alignment only)'],
        predictor_columns=FEATURE_COLUMNS, diagnostic_columns=['sequence_candidate_category_available'],
        excluded_predictors=['user', 'candidate_pos', 'wid', 'label', 'split', 'association_fold'],
        inputs=inputs, sources={'notebook': notebook_fingerprint()},
        environment=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, pyarrow=pa.__version__),
        cache_policy='Fresh rebuild on every run; no intermediate cache reused', artifacts=artifacts)
    save_json(manifest, manifest_path)
    print(f'Completed in {report["elapsed_seconds"]:.1f}s; outputs: {out_dir}', flush=True)
    return report


## 5. Correctness checks on small examples

These 14 checks live in the notebook and use the functions above. They validate ages, episode and delay boundaries, support/smoothing, missing inputs, source selection, label independence, candidate-order invariance, metadata conflicts and lossless IDs. A tiny raw-TXT fixture also checks candidate duplicates, existing split reuse, cross-fitting and nonzero scores. Temporary fixture files are removed automatically; the real data is not modified.


In [16]:
class CalculationTests(unittest.TestCase):
    def test_ages_and_invalid_gaps(self):
        np.testing.assert_allclose(reconstruct_ages([0, 12, 4, 5], 3) * 86400, [21, 9, 5])
        self.assertEqual(len(reconstruct_ages([0], 0)), 0)
        for gaps in [[1, 2], [0, -1], [0, math.nan], [0, math.inf], [0, 1, 2]]:
            with self.assertRaises(ValueError):
                reconstruct_ages(gaps, 1)

    def test_fixed_anchor_and_inclusive_episode_boundary(self):
        # 0, 20 and 40 minutes: the third purchase must start a NEW episode.
        ep = episodes([(1, 10), (1, 10 - 20 / 1440), (1, 10 - 40 / 1440)])
        self.assertEqual(len(ep), 2)
        self.assertAlmostEqual(ep[0][1], 10 - 20 / 1440)
        self.assertEqual(len(episodes([(1, 10), (1, 10 - 30 / 1440)])), 1)

    def test_equal_time_order_and_unknowns(self):
        self.assertEqual(episodes([(2, 10), (None, 10), (1, 10)]), [(2, 10), (1, 10)])
        with self.assertRaises(ValueError):
            episodes([(1, 1), (2, 2)])

    def test_window_boundaries_direction_and_followup(self):
        def has_pair(delay, age=10):
            return count_associations([[(1, age), (2, age - delay)]])['pairs']
        self.assertEqual(has_pair(1 / 24)[1, 2], 1)
        self.assertEqual(has_pair(7)[1, 2], 1)
        self.assertFalse(has_pair(1 / 24 - .00001))
        self.assertFalse(has_pair(7.00001))
        self.assertEqual(has_pair(1, age=8)[1, 2], 1)
        self.assertFalse(has_pair(1, age=7.99999))
        self.assertNotIn((2, 1), has_pair(1))
        self.assertFalse(count_associations([[(1, 10), (1, 9)]])['pairs'])

    def test_empty_windows_repeated_destinations_and_distinct_records(self):
        counts = count_associations([[(1, 20), (2, 19), (2, 18), (1, 10), (2, 9)], [(3, 20)]])
        self.assertEqual(counts['sources'][1], 2)
        self.assertEqual(counts['pairs'][1, 2], 2)
        self.assertEqual(counts['records'][1, 2], 1)
        self.assertEqual(counts['destinations'][2], 2)
        self.assertEqual(counts['sources'][3], 1)  # empty source window counted
        self.assertEqual(sum(counts['sources'].values()), 6)

    def test_smoothing_support_and_zero_background(self):
        counts = dict(sources=Counter({1: 50, 3: 50}), pairs=Counter({(1, 2): 20}),
                      destinations=Counter({2: 20}), records=Counter({(1, 2): 10}))
        row = list(association_rows(counts))[0]
        self.assertAlmostEqual(row['background_probability'], .2)
        self.assertAlmostEqual(row['probability'], 24 / 70)
        self.assertAlmostEqual(row['lift'], (24 / 70) / .2)
        self.assertTrue(row['supported'])
        for config in [replace(Config(), min_source_episodes=51), replace(Config(), min_pair_records=11)]:
            self.assertFalse(list(association_rows(counts, config))[0]['supported'])
        counts['destinations'].clear()
        self.assertEqual(list(association_rows(counts)), [])

    def test_nonpositive_relationship_excluded(self):
        rows = [dict(source_category=1, destination_category=2, lift=.5, pair_records=20, supported=False)]
        self.assertEqual(scoring_lookup(rows), {})

    def test_score_selected_support_age_and_recency(self):
        lookup = {(1, 3): (4, 10), (2, 3): (8, 30)}
        values = list(transform([(1, 20), (1, 1), (2, 10)], [3, 3], lookup))
        self.assertEqual(values[0], values[1])
        self.assertAlmostEqual(values[0][0], math.exp(-1 / 7) * math.log(4))
        self.assertAlmostEqual(values[0][1], math.log1p(10))
        self.assertEqual(values[0][2:], (1, 1))
        self.assertEqual(list(transform([(1, 30)], [3], lookup))[0][3], 1)
        self.assertEqual(list(transform([(1, 30.0001)], [3], lookup))[0][3], 0)

    def test_ties_and_safe_missing_cases(self):
        lookup = {(1, 3): (4, 10), (2, 3): (4, 20)}
        value = list(transform([(2, 1), (1, 1)], [3], lookup))[0]
        self.assertAlmostEqual(value[1], math.log1p(10))  # smallest source ID
        for orders, cats in [([], [3]), ([(None, 1)], [3]), ([(1, 1)], [None]), ([(1, 1)], [1])]:
            value = list(transform(orders, cats, lookup))[0]
            self.assertEqual((value[0], value[1], value[3]), (0, 0, 0))
            self.assertTrue(math.isnan(value[2]))

    def test_candidate_order_does_not_affect_values(self):
        lookup = {(1, 2): (4, 10), (1, 3): (2, 15)}
        a = list(transform([(1, 1)], [2, 3, 2], lookup))
        b = list(transform([(1, 1)], [3, 2, 2], lookup))
        self.assertEqual(a, [b[1], b[0], b[2]])

    def test_training_isolation(self):
        training = [[(1, 10), (2, 9)]]
        held_out = [[(3, 10), (4, 9)]]
        counts = count_associations(training)
        self.assertNotIn((3, 4), counts['pairs'])
        self.assertIn((3, 4), count_associations(training + held_out)['pairs'])


class IntegrationTests(unittest.TestCase):
    def test_raw_to_features_splits_folds_duplicates_and_label_independence(self):
        with tempfile.TemporaryDirectory() as tmp:
            root = Path(tmp)
            metadata = root / 'product_meta_data.txt'
            metadata.write_text('wid\tcate_id_2\n101\t1\n102\t2\n103\t3\n')
            behavior = root / 'user_behavior_data.txt'
            header = 'query\tcandidate_wid_list\tcandidate_label_list\thistory_qry_list\thistory_wid_list\thistory_type_list\thistory_time_list\n'
            # Two orders 10 and 9 days ago. Duplicate candidate occurrences kept.
            row = 'q\t102_102_999\t3_0_0\t-1_-1\t101_102\tORD_ORD\t0_86400_777600\n'
            behavior.write_text(header + row * 180)
            out = root / 'experiment'
            build_experiment(root, out, sample_size=180)
            frame = pd.read_parquet(out / 'candidate_features.parquet')
            membership = pd.read_parquet(out / 'sample_membership.parquet')
            expected = np.random.default_rng(42).choice(['train', 'val', 'test'], 180, p=[.7, .15, .15])
            self.assertEqual(membership['split'].tolist(), expected.tolist())
            self.assertEqual(len(frame), 540)
            self.assertTrue(frame['sequence_available'].any())
            self.assertFalse(frame.duplicated(['user', 'candidate_pos']).any())
            self.assertTrue((frame[frame['wid'] == 999]['sequence_candidate_category_available'] == 0).all())
            train = membership[membership['split'] == 'train']
            for fold in range(3):
                windows = pd.read_parquet(out / f'source_windows_fold_{fold}.parquet')
                self.assertTrue((windows['source_episodes'] == (train['association_fold'] != fold).sum()).all())
            full = pd.read_parquet(out / 'source_windows_all_train.parquet')
            self.assertTrue((full['source_episodes'] == len(train)).all())
            # Changing every candidate outcome changes no input feature or association.
            before = frame.drop(columns='label')
            behavior.write_text(header + row.replace('3_0_0', '0_2_1') * 180)
            build_experiment(root, out, sample_size=180)
            pd.testing.assert_frame_equal(before, pd.read_parquet(out / 'candidate_features.parquet').drop(columns='label'))
            self.assertTrue((out / 'manifest.json').is_file())

    def test_metadata_conflicts_and_lossless_ids(self):
        with tempfile.TemporaryDirectory() as tmp:
            path = Path(tmp) / 'meta.txt'
            large = 9007199254740993
            path.write_text(f'wid\tcate_id_2\n{large}\t1\n{large}\t2\n')
            with self.assertRaises(ValueError):
                load_metadata(path, np.array([large], dtype=np.int64))
            path.write_text(f'wid\tcate_id_2\n{large}\t1\n')
            self.assertEqual(load_metadata(path, np.array([large], dtype=np.int64))[0], {large: 1})

    def test_existing_split_reused_and_misalignment_rejected(self):
        with tempfile.TemporaryDirectory() as tmp:
            p = Path(tmp) / 'features.parquet'
            pd.DataFrame(dict(user=[1, 1], wid=[10, 11], label=[0, 3], split=['test', 'test'])).to_parquet(p)
            record = dict(user=1, candidates=[10, 11], labels=[0, 3], split='train')
            use_existing_splits([record], p)
            self.assertEqual(record['split'], 'test')
            record['candidates'] = [11, 10]
            with self.assertRaises(ValueError):
                use_existing_splits([record], p)


In [17]:
suite = unittest.TestSuite()
for test_class in [CalculationTests, IntegrationTests]:
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(test_class))
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful(), 'Correctness checks failed; do not build the sample.'


Fingerprinting raw inputs...
Selected 180/180 records; joining 3 product IDs
fold_0: {'eligible_windows': 172, 'observed_pairs': 1, 'supported_positive_pairs': 1}
fold_1: {'eligible_windows': 172, 'observed_pairs': 1, 'supported_positive_pairs': 1}
fold_2: {'eligible_windows': 172, 'observed_pairs': 1, 'supported_positive_pairs': 1}
all_train: {'eligible_windows': 258, 'observed_pairs': 1, 'supported_positive_pairs': 1}
Completed in 0.0s; outputs: /var/folders/0h/vlwccjz102g0zft6p9kyx91w0000gn/T/tmpew5wcn5e/experiment
Fingerprinting raw inputs...
Selected 180/180 records; joining 3 product IDs
fold_0: {'eligible_windows': 172, 'observed_pairs': 1, 'supported_positive_pairs': 1}
fold_1: {'eligible_windows': 172, 'observed_pairs': 1, 'supported_positive_pairs': 1}
fold_2: {'eligible_windows': 172, 'observed_pairs': 1, 'supported_positive_pairs': 1}
all_train: {'eligible_windows': 258, 'observed_pairs': 1, 'supported_positive_pairs': 1}
Completed in 0.0s; outputs: /var/folders/0h/vlwccjz1

test_ages_and_invalid_gaps (__main__.CalculationTests.test_ages_and_invalid_gaps) ... ok
test_candidate_order_does_not_affect_values (__main__.CalculationTests.test_candidate_order_does_not_affect_values) ... ok
test_empty_windows_repeated_destinations_and_distinct_records (__main__.CalculationTests.test_empty_windows_repeated_destinations_and_distinct_records) ... ok
test_equal_time_order_and_unknowns (__main__.CalculationTests.test_equal_time_order_and_unknowns) ... ok
test_fixed_anchor_and_inclusive_episode_boundary (__main__.CalculationTests.test_fixed_anchor_and_inclusive_episode_boundary) ... ok
test_nonpositive_relationship_excluded (__main__.CalculationTests.test_nonpositive_relationship_excluded) ... ok
test_score_selected_support_age_and_recency (__main__.CalculationTests.test_score_selected_support_age_and_recency) ... ok
test_smoothing_support_and_zero_background (__main__.CalculationTests.test_smoothing_support_and_zero_background) ... ok
test_ties_and_safe_missing_cases (

## 6. Build and save the 20,000-record sample

`REBUILD=True` performs the full calculation from TXT. Set it to `False` to inspect an existing completed run after verifying its input, notebook-code and artifact hashes, configuration and dependency versions. Do not run concurrent builds in the same output directory.


In [18]:
if REBUILD:
    report = build_experiment(DATA_DIR, OUT_DIR, sample_size=SAMPLE_SIZE, seed=SEED)
else:
    manifest = json.loads((OUT_DIR / 'manifest.json').read_text())
    assert manifest['sample_size'] == SAMPLE_SIZE and manifest['sample_seed'] == SEED
    assert manifest['config'] == asdict(Config())
    assert manifest['environment'] == dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, pyarrow=pa.__version__)
    for group in ['inputs', 'sources', 'artifacts']:
        for saved in manifest[group].values():
            actual = notebook_fingerprint(Path(saved['path'])) if saved.get('kind') == 'notebook_code' else fingerprint(Path(saved['path']))
            assert actual == saved, saved['path']
    report = json.loads((OUT_DIR / 'diagnostics.json').read_text())
manifest = json.loads((OUT_DIR / 'manifest.json').read_text())
print('Split policy:', manifest['split_policy'])
print('Elapsed seconds:', round(report['elapsed_seconds'], 2))
print('Process peak RSS (GiB):', round(report['process_peak_rss_bytes'] / 2**30, 3))
print('Selected records:', report['selected_records'])
print('Selected order events:', report['selected_order_events'])


Fingerprinting raw inputs...
Selected 20,000/173,831 records; joining 1,766,452 product IDs
fold_0: {'eligible_windows': 240332, 'observed_pairs': 47381, 'supported_positive_pairs': 4660}
fold_1: {'eligible_windows': 246434, 'observed_pairs': 48545, 'supported_positive_pairs': 4725}
fold_2: {'eligible_windows': 243320, 'observed_pairs': 48459, 'supported_positive_pairs': 4618}
all_train: {'eligible_windows': 365043, 'observed_pairs': 55830, 'supported_positive_pairs': 6115}
Completed in 27.1s; outputs: /Users/chounrayhan/Desktop/Repos/BT4222/BT4222-Group7/Raw Data/processed/purchase_sequence_affinity_sample
Split policy: Step 2 seed=42, 70/15/15 draw across all original record IDs
Elapsed seconds: 27.12
Process peak RSS (GiB): 0.891
Selected records: 20000
Selected order events: 785868


## 7. Coverage, within-slate variation and distributions

Report all complete sampled sessions, including sessions without positive labels. Availability is not predictive accuracy. Variation within a slate is necessary for this scalar score to change that slate's ranking by itself. Cross-fitting uses less fitting data than the final table, so training and held-out coverage may differ.


In [19]:
fits = pd.DataFrame(report['fits']).T
display(fits)
coverage_columns = ['sessions', 'candidate_rows', 'order_events', 'order_category_coverage',
                    'candidate_category_coverage', 'candidate_availability',
                    'session_availability', 'session_score_variation']
coverage = pd.DataFrame({s: {k: d[k] for k in coverage_columns} for s, d in report['by_split'].items()}).T
display(coverage)
for split, values in report['by_split'].items():
    print(split)
    display(pd.DataFrame(values['feature_summary']))
    print('Missing:', values['missing_values'])


train
Missing: {'sequence_score': 0, 'sequence_support_log': 0, 'sequence_source_age_days': 760091, 'sequence_available': 0}
val
Missing: {'sequence_score': 0, 'sequence_support_log': 0, 'sequence_source_age_days': 163946, 'sequence_available': 0}
test
Missing: {'sequence_score': 0, 'sequence_support_log': 0, 'sequence_source_age_days': 154931, 'sequence_available': 0}


,eligible_windows,observed_pairs,supported_positive_pairs
fold_0,240332,47381,4660
fold_1,246434,48545,4725
fold_2,243320,48459,4618
all_train,365043,55830,6115


,sessions,candidate_rows,order_events,order_category_coverage,candidate_category_coverage,candidate_availability,session_availability,session_score_variation
train,14019.0,1249132.0,555859.0,0.911380,0.995446,0.391505,0.508167,0.437264
val,3018.0,272207.0,114896.0,0.912756,0.995852,0.397716,0.508615,0.433068
test,2963.0,260639.0,115113.0,0.908724,0.995377,0.405572,0.513331,0.439757


,sequence_score,sequence_support_log,sequence_source_age_days,sequence_available
count,1.249132e+06,1.249132e+06,489041.000000,1.249132e+06
mean,8.548246e-02,1.513070e+00,10.693885,3.915047e-01
std,2.299490e-01,1.981285e+00,8.533828,4.880870e-01
min,0.000000e+00,0.000000e+00,0.000012,0.000000e+00
25%,0.000000e+00,0.000000e+00,3.236632,0.000000e+00
50%,0.000000e+00,0.000000e+00,8.628391,0.000000e+00
75%,4.814213e-02,3.367296e+00,17.020243,1.000000e+00
max,3.063299e+00,6.533789e+00,29.999363,1.000000e+00


,sequence_score,sequence_support_log,sequence_source_age_days,sequence_available
count,272207.000000,272207.000000,108261.000000,272207.000000
mean,0.083474,1.630949,10.685004,0.397716
std,0.235274,2.112573,8.438557,0.489427
min,0.000000,0.000000,0.000012,0.000000
25%,0.000000,0.000000,3.349468,0.000000
50%,0.000000,0.000000,8.902338,0.000000
75%,0.050853,3.637586,17.083414,1.000000
max,3.160522,6.900731,29.940325,1.000000


,sequence_score,sequence_support_log,sequence_source_age_days,sequence_available
count,260639.000000,260639.000000,105708.000000,260639.000000
mean,0.083169,1.682127,10.620512,0.405572
std,0.221872,2.143018,8.335949,0.491003
min,0.000000,0.000000,0.000012,0.000000
25%,0.000000,0.000000,3.485394,0.000000
50%,0.000000,0.000000,8.881713,0.000000
75%,0.050236,3.688879,16.948311,1.000000
max,3.489117,6.900731,29.994432,1.000000


In [20]:
# Candidate occurrence alignment and predictor separation.
features = pd.read_parquet(OUT_DIR / 'candidate_features.parquet')
assert not features.duplicated(['user', 'candidate_pos']).any()
assert features.groupby('user')['split'].nunique().max() == 1
assert not set(FEATURE_COLUMNS) & set(manifest['excluded_predictors'])
display(features.loc[features.sequence_available == 1].head(10))
print('Saved files:')
print('\n'.join(p.name for p in sorted(OUT_DIR.iterdir())))


Saved files:
associations_all_train.parquet
associations_fold_0.parquet
associations_fold_1.parquet
associations_fold_2.parquet
candidate_features.parquet
diagnostics.json
manifest.json
sample_membership.parquet
source_windows_all_train.parquet
source_windows_fold_0.parquet
source_windows_fold_1.parquet
source_windows_fold_2.parquet


,user,candidate_pos,wid,label,split,association_fold,sequence_candidate_category_available,sequence_score,sequence_support_log,sequence_source_age_days,sequence_available
323,68,1,89425058,0,train,2,1,0.007798,2.995732,28.763866,1
326,68,4,67349654,0,train,2,1,0.004377,3.258096,28.763866,1
328,68,6,20174739,0,train,2,1,0.007798,2.995732,28.763866,1
330,68,8,79943573,0,train,2,1,0.007798,2.995732,28.763866,1
332,68,10,73197026,0,train,2,1,0.007798,2.995732,28.763866,1
333,68,11,65530319,0,train,2,1,0.007798,2.995732,28.763866,1
334,68,12,28079823,0,train,2,1,0.004377,3.258096,28.763866,1
335,68,13,3857862,0,train,2,1,0.007798,2.995732,28.763866,1
336,68,14,18178337,0,train,2,1,0.004377,3.258096,28.763866,1
337,68,15,15042945,0,train,2,1,0.007798,2.995732,28.763866,1


## 8. Takeaways and next milestone

See `reports/purchase_sequence_affinity_sample.md` for measured results. This notebook establishes construction feasibility, coverage and tested calculation semantics. It does not establish ranking improvement. Candidates in the same destination category share a score, so inspect within-slate variation as well as availability.

A later evaluation should compare the existing LightGBM baseline with versus without the four predictor columns, preserving candidates/splits and selecting parameters on validation data. Keep the existing `cat3_cross_lift_max` in both versions initially. Model importance, warm/cold segment analysis and feature audit remain deferred.

Unlike Step 4b's population transition table, these associations are training-only, cross-fitted for training rows, and use timed level-2 purchase episodes and recency decay, as required by the handoff.
